# Midterm Project: Mini Data Science Project
### Student Habits vs Academic Performance

**Research question:** Which daily habits best predict a student's exam score, and what types of students exist?

**Techniques used:** Linear Regression and K-means Clustering

**How to use this notebook:** put `student_habits_performance.csv` in the same folder, then run the cells from top to bottom (Shift + Enter).

## 1. Dataset Selection

- **Dataset:** Student Habits vs Academic Performance (Kaggle)
- **Rows:** about 1,000 students
- **Target variable:** `exam_score`
- **Features:** study hours, social media, Netflix, sleep, attendance, diet, exercise, mental health, parental education, and more

**Why this dataset?** It is relatable to our class, clean enough to analyze quickly, and has a numeric target for regression plus habit variables for clustering.

## 2. Setup

Import the libraries and set the shared options. `save` stores each chart in a `figures` folder so you can drop it into your slides.

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error, silhouette_score
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="deep")
RANDOM_STATE = 42
TARGET = "exam_score"
os.makedirs("figures", exist_ok=True)

def save(name):
    plt.tight_layout()
    plt.savefig(os.path.join("figures", name), dpi=200)
    plt.show()

## 3. Load the Data

Read the CSV file and take a first look at its size, columns, and data types.

In [ ]:
CSV_PATH = "student_habits_performance.csv"
df = pd.read_csv(CSV_PATH)
print("Shape:", df.shape)
df.head()

In [ ]:
df.info()

## 4. Data Preparation

Before analysis we need to make sure the data is clean. The steps are:

1. **Check** for missing values and duplicate rows
2. **Drop** the `student_id` column because it is only an identifier
3. **Fill** missing numbers with the median and missing categories with `Unknown`
4. **Handle outliers** using the IQR rule, clipping extreme feature values

In [ ]:
missing = df.isna().sum()
print("Missing values:")
print(missing[missing > 0] if missing.any() else "None")
print("\nDuplicate rows:", df.duplicated().sum())

In [ ]:
df = df.drop(columns=["student_id"], errors="ignore")
df = df.drop_duplicates().dropna(subset=[TARGET])

num_cols = df.select_dtypes(include="number").columns.drop(TARGET).tolist()
cat_cols = df.select_dtypes(exclude="number").columns.tolist()

for c in num_cols:
    df[c] = df[c].fillna(df[c].median())
for c in cat_cols:
    df[c] = df[c].fillna("Unknown")

print("Numeric features:", num_cols)
print("Categorical features:", cat_cols)

In [ ]:
outliers = {}
for c in num_cols:
    q1, q3 = df[c].quantile([0.25, 0.75])
    low, high = q1 - 1.5 * (q3 - q1), q3 + 1.5 * (q3 - q1)
    outliers[c] = int(((df[c] < low) | (df[c] > high)).sum())
    df[c] = df[c].clip(low, high)

print("Outliers clipped per column:")
print(pd.Series(outliers))
print("\nClean shape:", df.shape)

## 5. Descriptive Statistics and Visualizations

Before modeling, we describe the data. This shows what a typical student looks like and which variables might matter.

Replace the notes below with what you see in your own charts.

In [ ]:
df.describe().round(2)

In [ ]:
for c in cat_cols:
    print(df[c].value_counts(), "\n")

### 5.1 Distribution of exam scores

Shows how scores are spread. Look for the center, the spread, and any skew.

In [ ]:
plt.figure(figsize=(7, 4))
sns.histplot(df[TARGET], bins=25, kde=True)
plt.title("Distribution of Exam Scores")
plt.xlabel("Exam score")
save("01_exam_score_distribution.png")

### 5.2 Distribution of the numeric features

Shows the shape of each habit variable, such as study hours and sleep.

In [ ]:
df[num_cols].hist(figsize=(12, 8), bins=20, edgecolor="white")
plt.suptitle("Distributions of Numeric Features", y=1.02)
save("02_numeric_histograms.png")

### 5.3 Correlation heatmap

Values near **+1** mean two variables rise together, near **-1** means one rises as the other falls, and near **0** means little linear relationship. Focus on the `exam_score` row.

In [ ]:
plt.figure(figsize=(9, 7))
sns.heatmap(df[num_cols + [TARGET]].corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation Heatmap")
save("03_correlation_heatmap.png")

### 5.4 Study hours vs exam score

The headline relationship of the project. The red line is the best straight-line fit.

In [ ]:
plt.figure(figsize=(7, 5))
sns.regplot(data=df, x="study_hours_per_day", y=TARGET,
            scatter_kws={"alpha": 0.4}, line_kws={"color": "red"})
plt.title("Study Hours per Day vs Exam Score")
save("04_study_hours_vs_score.png")

### 5.5 Exam score by category

Compares score distributions across groups such as diet quality or internet quality.

In [ ]:
show = [c for c in ["diet_quality", "parental_education_level", "part_time_job", "internet_quality"] if c in df.columns]
fig, axes = plt.subplots(1, len(show), figsize=(5 * len(show), 4))
for ax, c in zip(np.atleast_1d(axes), show):
    sns.boxplot(data=df, x=c, y=TARGET, ax=ax)
    ax.set_title(f"Exam score by {c}")
    ax.tick_params(axis="x", rotation=30)
save("05_boxplots_by_category.png")

## 6. Analysis Technique 1: Linear Regression

**Goal:** predict `exam_score` from habits and find which habits matter most.

**Why linear regression?** The target is a number, and regression gives interpretable coefficients that show how much each habit changes the score.

**Steps**
1. Convert text categories into numeric columns (one-hot encoding)
2. Split into 80% training and 20% test data
3. Standardize features so coefficients are comparable
4. Fit the model and measure accuracy on the test data

In [ ]:
X = pd.get_dummies(df.drop(columns=[TARGET]), drop_first=True).astype(float)
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

lr = LinearRegression().fit(X_train_s, y_train)
pred = lr.predict(X_test_s)

### 6.1 Model performance

- **R²:** share of score variation the model explains (1.0 is perfect)
- **RMSE and MAE:** typical prediction error, in exam points
- **Cross-validated R²:** a more reliable estimate across 5 different splits

In [ ]:
r2 = r2_score(y_test, pred)
rmse = float(np.sqrt(mean_squared_error(y_test, pred)))
mae = mean_absolute_error(y_test, pred)
cv_r2 = cross_val_score(LinearRegression(), scaler.transform(X), y, cv=5, scoring="r2").mean()

pd.DataFrame({"Metric": ["R2 (test)", "R2 (5-fold CV)", "RMSE", "MAE"],
              "Value": [r2, cv_r2, rmse, mae]}).round(3)

### 6.2 Actual vs predicted

Points close to the red dashed line mean accurate predictions.

In [ ]:
plt.figure(figsize=(6, 6))
plt.scatter(y_test, pred, alpha=0.6)
lims = [min(y_test.min(), pred.min()), max(y_test.max(), pred.max())]
plt.plot(lims, lims, "r--", label="Perfect prediction")
plt.xlabel("Actual exam score")
plt.ylabel("Predicted exam score")
plt.title(f"Actual vs Predicted (R2 = {r2:.2f})")
plt.legend()
save("06_actual_vs_predicted.png")

### 6.3 Residuals

Residuals are the prediction errors. A good model has them centered at zero with no strong skew.

In [ ]:
plt.figure(figsize=(7, 4))
sns.histplot(y_test - pred, bins=25, kde=True)
plt.title("Residual Distribution")
plt.xlabel("Residual")
save("07_residuals.png")

### 6.4 What drives exam scores?

Because features were standardized, bigger bars mean a stronger effect. **Blue** raises the score and **red** lowers it.

In [ ]:
coefs = pd.Series(lr.coef_, index=X.columns).sort_values(key=np.abs, ascending=False)
top = coefs.head(10).sort_values()

plt.figure(figsize=(8, 5))
plt.barh(top.index, top.values, color=["tab:red" if v < 0 else "tab:blue" for v in top.values])
plt.title("What Drives Exam Scores? (Standardized Coefficients)")
plt.xlabel("Effect on exam score")
save("08_feature_importance.png")

coefs.head(10).round(3)

## 7. Analysis Technique 2: K-means Clustering

**Goal:** group students into learner profiles using habits only, then check how scores differ between groups.

**Why K-means?** It groups similar students together without using the exam score, so any score differences between groups are a genuine finding rather than something we forced.

**Steps**
1. Select habit features and standardize them
2. Choose the number of clusters (k) using the elbow method and silhouette score
3. Fit K-means and profile each cluster

In [ ]:
features = [c for c in ["study_hours_per_day", "social_media_hours", "netflix_hours",
                        "attendance_percentage", "sleep_hours", "exercise_frequency",
                        "mental_health_rating"] if c in df.columns] or num_cols
Xc = StandardScaler().fit_transform(df[features])
features

### 7.1 Choosing k

- **Elbow plot:** pick the k where the curve starts to flatten
- **Silhouette score:** higher means better separated clusters

We prefer 3 or 4 clusters because they are easier to explain in a presentation.

In [ ]:
ks = list(range(2, 9))
inertias, sils = [], []
for k in ks:
    model = KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE).fit(Xc)
    inertias.append(model.inertia_)
    sils.append(silhouette_score(Xc, model.labels_))

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(ks, inertias, "o-")
ax[0].set_title("Elbow Method")
ax[0].set_xlabel("k")
ax[0].set_ylabel("Inertia")
ax[1].plot(ks, sils, "o-", color="green")
ax[1].set_title("Silhouette Score")
ax[1].set_xlabel("k")
save("09_choose_k.png")

In [ ]:
best_k = ks[int(np.argmax(sils))]
K = best_k if 3 <= best_k <= 4 else 3
print("Best k by silhouette:", best_k)
print("k used:", K)

km = KMeans(n_clusters=K, n_init=10, random_state=RANDOM_STATE).fit(Xc)
df["cluster"] = km.labels_
sil = silhouette_score(Xc, km.labels_)
print("Silhouette score:", round(sil, 3))

### 7.2 Cluster profiles

Average habits and exam score for each cluster. Use this table to **name each group** (for example "Focused Achievers" or "Distracted Night Owls").

In [ ]:
profile = df.groupby("cluster")[features + [TARGET]].mean().round(2)
profile["students"] = df["cluster"].value_counts().sort_index()
profile

### 7.3 Clusters in two dimensions

PCA squeezes all habit variables into two axes so the clusters can be plotted. Overlap between groups is normal for behavioral data.

In [ ]:
pcs = PCA(n_components=2, random_state=RANDOM_STATE).fit_transform(Xc)
plt.figure(figsize=(7, 5))
sns.scatterplot(x=pcs[:, 0], y=pcs[:, 1], hue=df["cluster"], palette="Set2", s=40)
plt.title("Student Clusters (PCA projection)")
plt.xlabel("PC1")
plt.ylabel("PC2")
save("10_clusters_pca.png")

### 7.4 Cluster centers

Red means above average and blue means below average for that habit. This is the easiest chart for describing each profile.

In [ ]:
plt.figure(figsize=(9, 3 + K * 0.4))
sns.heatmap(pd.DataFrame(km.cluster_centers_, columns=features), annot=True, fmt=".2f", cmap="RdBu_r", center=0)
plt.title("Cluster Centers (standardized)")
plt.ylabel("Cluster")
save("11_cluster_centers.png")

### 7.5 Exam score by cluster

The key check: do the habit-based groups really differ in exam performance?

In [ ]:
plt.figure(figsize=(6, 4))
sns.boxplot(data=df, x="cluster", y=TARGET, palette="Set2")
plt.title("Exam Score by Student Profile")
save("12_score_by_cluster.png")

## 8. Analysis and Insights

Run the cell below to print the main findings, then rewrite them in your own words.

**Questions to answer in your presentation**
- Which habit has the strongest effect on exam scores?
- Do habits with a negative effect (like social media) match what you expected?
- How do the K-means groups compare with the regression findings?
- What was the most surprising result?

**Limitations to mention**
- The dataset is small and may be synthetic
- Regression shows association, not proof of cause
- Clusters overlap, so group boundaries are fuzzy

In [ ]:
best_num = df[num_cols + [TARGET]].corr()[TARGET].drop(TARGET).abs().idxmax()
best_cluster = profile[TARGET].idxmax()
worst_cluster = profile[TARGET].idxmin()

print(f"1. The regression explains about {r2 * 100:.0f}% of the variation in exam scores (typical error +/- {mae:.1f} points).")
print(f"2. Strongest drivers: {', '.join(coefs.head(3).index)}.")
print(f"3. Strongest numeric link to score: {best_num} (r = {df[best_num].corr(df[TARGET]):.2f}).")
print(f"4. K-means found {K} student profiles (silhouette {sil:.2f}).")
print(f"5. Cluster {best_cluster} scores highest ({profile[TARGET].max():.1f}) and cluster {worst_cluster} lowest ({profile[TARGET].min():.1f}).")

## 9. Presentation Guide (5 to 10 minutes)

| Slide | Content | Charts to use |
|---|---|---|
| 1 | Title and research question | none |
| 2 | Dataset overview | none |
| 3 | Data preparation steps | none |
| 4 | Descriptive statistics | 01, 02, 04, 05 |
| 5 | Correlation heatmap | 03 |
| 6 | Linear regression performance | 06, 07 |
| 7 | What drives exam scores | 08 |
| 8 | Choosing k for K-means | 09 |
| 9 | Student profiles | 10, 11, 12 |
| 10 | Surprising finding, conclusion, limitations | none |

All charts are saved in the `figures` folder.

**Recording tips**
- Aim for about 30 to 40 seconds per slide
- Show one chart per idea and say what it means in one sentence
- End with one clear takeaway
- Do a short test recording to check your audio first